# Loan Approval Dataset: Data Understanding and Preprocessing

This notebook performs a complete data understanding and preprocessing workflow for coursework submission.

Steps included:
1. Load data from CSV
2. Inspect shape, columns, and sample rows
3. Explore summary statistics, missing values, and data types
4. Drop unnecessary ID-like columns
5. Impute missing values (mean for numeric, mode for categorical)
6. Encode categorical variables
7. Build classification and regression datasets
8. Show target value counts and plot target distribution

## 1. Import Required Libraries
Import the libraries needed for data handling, preprocessing, and visualization.

In [ ]:
# Import libraries for data work, plots, and encoding.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
# Set one simple plotting style for consistency.
sns.set_style("whitegrid")

## 2. Load Dataset and Inspect Basic Structure
Load the CSV file, then display dataset shape, column names, and the first rows.

In [ ]:
# Load the dataset from CSV.
csv_path = "loan_approval_data.csv"
df = pd.read_csv(csv_path)

# Basic check: shape, columns, and sample rows.
print("Dataset shape:", df.shape)
print("\nColumn names:")
display(pd.DataFrame({"column": df.columns}))
print("\nFirst 5 rows:")
display(df.head())

## 3. Perform Data Exploration
Generate descriptive statistics, check missing values, and inspect data types.

In [ ]:
# Show summary stats for numeric variables.
print("Descriptive statistics (numeric):")
display(df.describe())

# Show current data types for all columns.
print("\nData types:")
display(df.dtypes)

# Check missing values before any cleaning.
print("\nMissing values (before cleaning):")
missing_before = df.isnull().sum()
display(missing_before[missing_before > 0].sort_values(ascending=False))

## 4. Drop Unnecessary Columns (Such as ID)
Automatically detect ID-like columns and remove them from the working dataset.

In [ ]:
# Helper to standardize column names for matching.
def normalize_name(text):
    return str(text).strip().lower().replace(" ", "").replace("_", "")

# Find columns that look like IDs.
id_like_columns = []
for col in df.columns:
    normalized_col = normalize_name(col)
    if normalized_col == "id" or normalized_col.endswith("id") or normalized_col in {"loanid", "applicationid"}:
        id_like_columns.append(col)

# Drop ID-like columns because they are not useful for modelling.
print("Detected ID-like columns:", id_like_columns)
df = df.drop(columns=id_like_columns, errors="ignore")

# Show remaining columns after feature selection step.
print("\nColumns after feature selection:")
display(pd.DataFrame({"column": df.columns}))

## 5. Handle Missing Values
Fill missing numeric values with mean and categorical values with mode.

In [ ]:
# Split columns by type for cleaning.
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = df.select_dtypes(exclude=[np.number]).columns.tolist()

# Show missing values before filling.
print("Missing values before cleaning:")
display(df.isnull().sum())

# Fill numeric missing values with mean.
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].mean())

# Fill categorical missing values with mode.
for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode(dropna=True)[0])

# Try converting categorical columns to numeric if possible.
dtype_changes = []
for col in categorical_columns:
    converted = pd.to_numeric(df[col], errors="coerce")
    if converted.notna().sum() == len(df):
        df[col] = converted
        dtype_changes.append(col)

print("\nColumns converted to numeric:", dtype_changes if dtype_changes else "None")

# Handle outliers using IQR capping.
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
outlier_summary = []
for col in numeric_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    before_count = ((df[col] < lower) | (df[col] > upper)).sum()
    if before_count > 0:
        df[col] = df[col].clip(lower=lower, upper=upper)
    outlier_summary.append({"column": col, "outliers_capped": int(before_count)})

print("\nOutlier handling summary (IQR capping):")
display(pd.DataFrame(outlier_summary))

# Final checks after cleaning.
print("\nMissing values after cleaning:")
display(df.isnull().sum())

print("\nData types after cleaning:")
display(df.dtypes)

## 6. Identify Target Columns and Prepare for Modeling
Find the classification target (Loan Approval Status) and regression target (Maximum Loan Amount) safely across possible naming variations.

In [ ]:
# Helper to find column names from multiple possible options.
def find_column_name(column_list, candidate_names):
    normalized_to_original = {
        str(col).strip().lower().replace(" ", "").replace("_", ""): col
        for col in column_list
    }
    for candidate in candidate_names:
        normalized_candidate = str(candidate).strip().lower().replace(" ", "").replace("_", "")
        if normalized_candidate in normalized_to_original:
            return normalized_to_original[normalized_candidate]
    return None

# Candidate names for classification and regression targets.
class_target_candidates = ["loan_approval_status", "Loan Approval Status", "loan_status", "approved"]
reg_target_candidates = ["max_allowed_loan", "Maximum Loan Amount", "maximum_loan_amount", "loan_amount_max"]

# Detect target columns from the current dataset.
class_target_col = find_column_name(df.columns, class_target_candidates)
reg_target_col = find_column_name(df.columns, reg_target_candidates)

# Stop with clear errors if targets are not found.
if class_target_col is None:
    raise ValueError("Could not find the classification target column (Loan Approval Status).")
if reg_target_col is None:
    raise ValueError("Could not find the regression target column (Maximum Loan Amount).")

# Print chosen target columns.
print("Classification target column:", class_target_col)
print("Regression target column:", reg_target_col)

## 7. Build Classification Dataset
Create `X_class` and `y_class`, encode categorical features using one-hot encoding, and encode target labels when needed.

In [ ]:
# Keep only feature columns (exclude both target columns).
selected_feature_columns = [
    col for col in df.columns
    if col not in {class_target_col, reg_target_col}
 ]

# Build classification feature matrix and target vector.
X_class = df[selected_feature_columns].copy()
y_class = df[class_target_col].copy()

# Encode categorical feature columns with one-hot encoding.
X_class = pd.get_dummies(X_class, drop_first=False)
if not pd.api.types.is_numeric_dtype(y_class):
    label_encoder = LabelEncoder()
    y_class = pd.Series(label_encoder.fit_transform(y_class), name=class_target_col)

# Show final shape and class distribution.
print("X_class shape:", X_class.shape)
print("y_class shape:", y_class.shape)
print("\nLoan Approval Status counts:")
display(y_class.value_counts(dropna=False).sort_index())

## 8. Plot Distribution of Loan Approval Status
Visualize the target distribution with a bar chart.

In [ ]:
# Plot target distribution for the classification task.
plt.figure(figsize=(8, 5))
sns.countplot(x=y_class.astype(str), color="#4c72b0")
plt.title("Target Distribution: Loan Approval Status")
plt.xlabel("Loan Approval Status")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 9. Build Regression Dataset (Approved Loans Only)
Filter only approved loans, then create `X_reg` and `y_reg` where the target is Maximum Loan Amount.

In [ ]:
# Start with approval status column for filtering approved cases.
approval_series = df[class_target_col].copy()
approved_value = 1

# Handle text labels like approved/yes/true.
if not pd.api.types.is_numeric_dtype(approval_series):
    unique_values = [str(v).strip().lower() for v in approval_series.dropna().unique().tolist()]
    approval_candidates = ["approved", "approve", "yes", "y", "true", "1"]
    matched_values = [v for v in unique_values if v in approval_candidates]
    if len(matched_values) > 0:
        approved_value = matched_values[0]
    approval_series = approval_series.astype(str).str.strip().str.lower()

# Handle numeric labels by selecting the class with higher mean allowed loan.
if pd.api.types.is_numeric_dtype(df[class_target_col]):
    class_means = df.groupby(class_target_col)[reg_target_col].mean()
    approved_value = class_means.idxmax()
    approval_series = df[class_target_col]

# Filter only approved rows for regression dataset.
approved_df = df[approval_series == approved_value].copy()
if approved_df.shape[0] == 0:
    raise ValueError("No approved loans were found. Please inspect class labels.")

# Build regression features and target.
X_reg = approved_df.drop(columns=[reg_target_col, class_target_col], errors="ignore").copy()
y_reg = approved_df[reg_target_col].copy()
X_reg = pd.get_dummies(X_reg, drop_first=False)

# Print basic checks and previews.
print("Approved class used for regression:", approved_value)
print("Approved records for regression:", approved_df.shape[0])
print("X_reg shape:", X_reg.shape)
print("y_reg shape:", y_reg.shape)

print("\nX_class preview:")
display(X_class.head())
print("X_reg preview:")
display(X_reg.head())

## 10. Final Output Summary
At this point, the notebook has created:
- `X_class` and `y_class` for classification
- `X_reg` and `y_reg` for regression (approved loans only)

These datasets are now ready for model training in later coursework sections.